In [9]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os
from collections import Counter

# Konfigurasi path
DATA_PATH = '../../outputs/evaluation/ground_truth_final.csv'
LEXICON_PATH = '../../../kamus/inset_plus_politik.csv'
OUTPUT_DIR = '../../outputs/RSN'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [10]:
# 2.2 Load Data Ground Truth
df = pd.read_csv(DATA_PATH)

print(f"\n[DATA] Ground truth berhasil dimuat: {len(df)} tweet")
print(f"Kolom tersedia: {df.columns.tolist()}")

df.head()


[DATA] Ground truth berhasil dimuat: 450 tweet
Kolom tersedia: ['no', 'timestamp', 'teks', 'teks_processed', 'ground_truth_mean', 'ground_truth_scaled', 'sd_annotator', 'ground_truth_category']


,no,timestamp,teks,teks_processed,ground_truth_mean,ground_truth_scaled,sd_annotator,ground_truth_category
0,50,2016-12-24T13:54:06.000Z,Komisi IX Ketua Komisi IX DPR RI merekomendasi...,komisi IX ketua komisi IX DPR RI merekomendasi...,0.8,0.20,0.447214,positive
1,84,2016-12-21T08:47:51.000Z,Dapilku Rumahku Reses dan silaturahim bersama ...,dapilku rumahku reses dan silaturahim bersama ...,0.8,0.20,0.447214,positive
2,89,2016-12-20T18:32:07.000Z,mengharukan wakil rakyat trlalu banyak waktuny...,mengharukan wakil rakyat terlalu banyak waktun...,-2.8,-0.70,0.447214,negative
3,101,2016-12-20T04:39:14.000Z,Ketua dan Para Wakil Ketua DPR RI PERLUNYA RUU...,ketua dan para wakil ketua DPR RI PERLUNYA RUU...,0.8,0.20,0.447214,positive
4,121,2016-12-18T04:07:09.000Z,Viva Anggota DPR Minta TNI AU Investigasi Kece...,viva anggota DPR meminta TNI AU investigasi ke...,0.6,0.15,0.547723,positive


In [11]:
# 2.3 Load Leksikon Gabungan

df_inset = pd.read_csv(LEXICON_PATH)       
df_inset['kata'] = df_inset['kata'].astype(str).str.strip().str.lower()

# Dictionary lexicon untuk matching
inset_dict = dict(zip(df_inset['kata'], df_inset['skor'])) 

print(f"[LEKSIKON] InSet+Politik berhasil dimuat: {len(inset_dict)} entri.")

[LEKSIKON] InSet+Politik berhasil dimuat: 9855 entri.


In [12]:
# 2.4 Definisi Kategori Kata Fungsi (Untuk Ignore Function)
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}

KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}

PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}

PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

In [13]:
# 2.5 Fungsi Tokenisasi
def tokenize(text):
    """Tokenisasi sederhana berdasarkan spasi"""
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)

total_tokens = df['tokens'].str.len().sum()
print(f"\n[TOKENISASI] Selesai. Total token: {total_tokens:,}")


[TOKENISASI] Selesai. Total token: 8,282


In [14]:
# 2.6 Fungsi Lexicon Matching TANPA Perlakuan Fungsi
def match_lexicon_no_filter(tokens, lexicon):
    """
    Memisahkan token menjadi 2 kategori:
    1. Matched: Kata yang ada di leksikon (termasuk kata fungsi jika ada di leksikon)
    2. Unmatched: Kata yang TIDAK ada di leksikon
    
    Note: Tidak ada kategori 'ignored' karena semua kata diproses.
    """
    matched_words = []
    unmatched_words = []
    
    for token in tokens:
        token_lower = token.lower()
        
        # Cek apakah kata ada di leksikon
        if token_lower in lexicon:
            matched_words.append(token)
        else:
            unmatched_words.append(token)
            
    return matched_words, unmatched_words

In [15]:
# 2.7 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching RSN (Tanpa Filter Fungsi)...")

df[['matched_words', 'unmatched_words']] = pd.DataFrame(
    df['tokens'].apply(
        lambda x: match_lexicon_no_filter(x, inset_dict)
    ).tolist(),
    index=df.index
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching RSN (Tanpa Filter Fungsi)...
[INFO] Lexicon matching selesai.


In [16]:
# 2.8 Perhitungan Statistik Matching
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

# Hitung berapa banyak kata fungsi yang ternyata 'matched' (ada di leksikon)
function_words_in_lexicon = set([w for w in ALL_FUNCTION_WORDS if w in inset_dict])
matched_function_count = 0
for lst in df['matched_words']:
    for word in lst:
        if word.lower() in function_words_in_lexicon:
            matched_function_count += 1

print("\n[STATISTIK] Hasil Lexicon Matching (RSN - No Function Filter):")
print(f"Total kata             : {total_words:,}")
print(f"Matched di Leksikon    : {total_matched:,} ({(total_matched/total_words)*100:.2f}%)")
print(f"Unmatched              : {total_unmatched:,} ({(total_unmatched/total_words)*100:.2f}%)")
print("-" * 60)
print(f"Coverage Rate          : {(total_matched/total_words)*100:.2f}%")
print(f"\n[ANALISIS NOISE] Kata fungsi yang ter-match (potensial noise): {matched_function_count:,}")
if total_matched > 0:
    print(f"Persentase kata fungsi terhadap total matched: {(matched_function_count/total_matched)*100:.2f}%")


[STATISTIK] Hasil Lexicon Matching (RSN - No Function Filter):
Total kata             : 8,282
Matched di Leksikon    : 5,746 (69.38%)
Unmatched              : 2,536 (30.62%)
------------------------------------------------------------
Coverage Rate          : 69.38%

[ANALISIS NOISE] Kata fungsi yang ter-match (potensial noise): 541
Persentase kata fungsi terhadap total matched: 9.42%


In [18]:
# 2.9 Analisis Top Unmatched Words
print("\n[ANALISIS] Top 50 Kata Unmatched Paling Sering Muncul:")
print("-"*50)

all_unmatched = []
for lst in df['unmatched_words']:  
    all_unmatched.extend([w.lower() for w in lst])

unmatched_freq = Counter(all_unmatched)
top_50_unmatched = unmatched_freq.most_common(50)

print(f"{'Kata':<25} | {'Frekuensi':<10}")
print("-"*40)
for word, freq in top_50_unmatched:
    print(f"{word:<25} | {freq:<10}")


[ANALISIS] Top 50 Kata Unmatched Paling Sering Muncul:
--------------------------------------------------
Kata                      | Frekuensi 
----------------------------------------
dan                       | 113       
di                        | 98        
?                         | 68        
ini                       | 65        
untuk                     | 59        
dengan                    | 43        
ke                        | 41        
!                         | 39        
jika                      | 27        
bisa                      | 26        
akan                      | 25        
juga                      | 22        
menjadi                   | 20        
saat                      | 19        
tetapi                    | 17        
atau                      | 14        
melakukan                 | 13        
tak                       | 12        
terus                     | 12        
h                         | 12        
kenapa                    | 11   

In [19]:
# 2.10 Preview Hasil Matching
print("\n[PREVIEW] 5 Tweet Pertama:")

for i in range(min(5, len(df))):
    print(f"\n[Tweet {i+1}]")
    teks_preview = df['teks_processed'].iloc[i]
    if len(teks_preview) > 80:
        teks_preview = teks_preview[:80] + "..."
    print(f"Teks: {teks_preview}")
    print(f"  ✓ Matched   : {df['matched_words'].iloc[i][:10]}...") 
    print(f"  ✗ Unmatched : {df['unmatched_words'].iloc[i][:10]}...")


[PREVIEW] 5 Tweet Pertama:

[Tweet 1]
Teks: komisi IX ketua komisi IX DPR RI merekomendasikan kepada pem agar tinjau ulang k...
  ✓ Matched   : ['komisi', 'IX', 'ketua', 'komisi', 'IX', 'DPR', 'RI', 'merekomendasikan', 'kepada', 'agar']...
  ✗ Unmatched : ['pem']...

[Tweet 2]
Teks: dapilku rumahku reses dan silaturahim bersama anggota DPR RI pkbmembelarakyat
  ✓ Matched   : ['rumahku', 'reses', 'silaturahim', 'bersama', 'anggota', 'DPR', 'RI', 'pkbmembelarakyat']...
  ✗ Unmatched : ['dapilku', 'dan']...

[Tweet 3]
Teks: mengharukan wakil rakyat terlalu banyak waktunya untuk update disosmed hampir ti...
  ✓ Matched   : ['mengharukan', 'wakil', 'rakyat', 'terlalu', 'banyak', 'update', 'tidak', 'ada', 'waktu', 'update']...
  ✗ Unmatched : ['waktunya', 'untuk', 'disosmed', 'hampir', 'untuk', 'diwakilinya']...

[Tweet 4]
Teks: ketua dan para wakil ketua DPR RI PERLUNYA RUU PELAPORAN KEUANGAN UNTUK DUNIA BI...
  ✓ Matched   : ['ketua', 'para', 'wakil', 'ketua', 'DPR', 'RI', 'RUU', 'PELAPOR

In [20]:
# 2.11 Simpan Output Matching
output_path = os.path.join(OUTPUT_DIR, 'rsn_lexicon_matching_tanpa_perlakuan.csv')
df.to_csv(output_path, index=False)

print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")
print("\n[CATATAN PENTING]")
print("Tahap ini HANYA untuk matching dan statistik.")
print("Semua kata yang match (termasuk fungsi) akan masuk ke perhitungan skor di Notebook Scoring berikutnya.")


[OUTPUT] Data berhasil disimpan ke: ../../outputs/RSN\rsn_lexicon_matching_tanpa_perlakuan.csv

[CATATAN PENTING]
Tahap ini HANYA untuk matching dan statistik.
Semua kata yang match (termasuk fungsi) akan masuk ke perhitungan skor di Notebook Scoring berikutnya.
